# Function Spaces and Norms

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 08.01–08.06, 09.01–09.06 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/11_Functional_Analysis/01_function_spaces_and_norms.ipynb)

---

## 🎯 Learning Objective

Learn Banach and Hilbert spaces, $L^p$ spaces, and bounded linear operators as the setting for functional analysis.

---

## 📐 Theory

`08.01` generalized the dot product to abstract inner products and mentioned, in passing, that
function-space inner products make Hilbert spaces the natural home for kernel methods. This
notebook makes that "function space" idea precise and general enough to describe what a neural
network actually *is*: not a formula, but a map between infinite-dimensional spaces of
functions.

### Norms, and why not every norm comes from an inner product

A **norm** $\|\cdot\|$ on a vector space generalizes "length": it must satisfy
$\|\mathbf{v}\|\ge 0$ (zero only for $\mathbf{v}=\mathbf{0}$), $\|c\mathbf{v}\|=|c|\|\mathbf{v}\|$,
and the **triangle inequality** $\|\mathbf{u}+\mathbf{v}\|\le\|\mathbf{u}\|+\|\mathbf{v}\|$. The
**$L^p$ norms** $\|\mathbf{v}\|_p = (\sum_i |v_i|^p)^{1/p}$ (or $\int |f|^p$ for functions) give
an entire family, one for each $p\ge 1$, all satisfying the triangle inequality — verified
numerically below. But only $p=2$ comes from an inner product ($\|\mathbf{v}\|_2 =
\sqrt{\langle\mathbf{v},\mathbf{v}\rangle}$): the tell-tale sign is the **parallelogram law**
$\|\mathbf{u}+\mathbf{v}\|^2+\|\mathbf{u}-\mathbf{v}\|^2 = 2\|\mathbf{u}\|^2+2\|\mathbf{v}\|^2$,
which holds *only* for $p=2$, confirmed below by direct counterexample at $p=1,3,\infty$. This
is exactly why $L^2$ gets to have angles, orthogonality, and projections (`08.01`'s whole
toolkit) while $L^1$ and $L^\infty$ — despite being perfectly good norms — do not.

### Completeness, and Banach vs. Hilbert spaces

A sequence is **Cauchy** if its terms eventually get arbitrarily close to *each other* (not
necessarily to any particular known point). A space is **complete** if every Cauchy sequence
actually converges to a limit *within that same space*. The rational numbers are the classic
counterexample: truncating $\sqrt{2}$'s decimal expansion produces a Cauchy sequence of rational
numbers (confirmed below) whose limit, $\sqrt{2}$, is irrational — the limit escapes the space.
A **Banach space** is a complete normed vector space; a **Hilbert space** is a complete
inner-product space (so: a Banach space whose norm happens to come from an inner product).
Completeness is what makes calculus on these spaces well-behaved: limits, derivatives, and
infinite sums of basis functions are guaranteed to land somewhere meaningful.

### $L^p$ function spaces

The $L^p$ spaces generalize the $\ell^p$ vector norms above to *functions*: $\|f\|_p =
\big(\int |f(x)|^p\,dx\big)^{1/p}$, and $L^p$ is the set of functions for which this integral is
finite. $L^2$ is the Hilbert space of functions with $\langle f,g\rangle = \int f(x)g(x)\,dx$ —
the exact structure `08.01` gestured at, and the natural setting for asking "how far apart are
two functions" or "does this sequence of approximating functions converge" in a way that
supports geometric reasoning (angles, projections, orthogonal decompositions) rather than just
raw distance.

### Bounded linear operators

A **linear operator** $T:V\to W$ between normed spaces is **bounded** if there's a constant $C$
with $\|T\mathbf{v}\|_W \le C\|\mathbf{v}\|_V$ for every $\mathbf{v}$ — it can't amplify length
by an unbounded factor. The smallest valid $C$ is the **operator norm** $\|T\| =
\sup_{\mathbf{v}\ne 0} \|T\mathbf{v}\|/\|\mathbf{v}\|$; for a matrix, this is exactly the largest
singular value (recall `01.07`), confirmed numerically below by comparing against many random
directions and against the theoretically exact maximizer. Every linear layer in a neural network
is a bounded linear operator in this precise sense, and its operator norm is exactly the
worst-case amplification factor that shows up in Lipschitz-based generalization bounds and
gradient-stability analyses (recall the vanishing/exploding-gradient discussions in `06`).

### Neural networks as operators between function spaces

A trained neural network $f_\theta$ approximating some target function $f^\star$ is best
understood, in this language, as one function living in $L^2$ trying to get close (in the
$L^2$ norm) to another. The **universal approximation theorem** says a sufficiently wide network
can get arbitrarily close to any reasonable $f^\star$ in this sense — not just at the training
points, but as measured by the actual $L^2$ distance $\|f_\theta - f^\star\|_2$ over the whole
input domain, verified concretely below. This function-space view is also the foundation of
**operator learning** (e.g. Fourier Neural Operators, DeepONet): instead of learning a function
$\mathbb{R}^n\to\mathbb{R}^m$, learn an *operator* mapping one entire function to another (e.g.
an initial condition to a PDE's solution), treating both input and output as points in a
function space rather than finite vectors.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting the "unit circle" (the set of vectors with norm exactly 1) for several different $L^p$
norms shows they're genuinely different shapes — and only the $p=2$ circle is round in the
usual sense, because only $p=2$ comes from an inner product.

In [ ]:
def p_unit_ball_boundary(p, n_points=400):
    theta = np.linspace(0, 2*np.pi, n_points)
    x_raw, y_raw = np.cos(theta), np.sin(theta)
    if p == np.inf:
        scale = np.maximum(np.abs(x_raw), np.abs(y_raw))
    else:
        scale = (np.abs(x_raw)**p + np.abs(y_raw)**p) ** (1/p)
    return x_raw / scale, y_raw / scale

fig, ax = plt.subplots(figsize=(7, 7))
colors = {1: "#C44E52", 2: "#4C72B0", 3: "#55A868", np.inf: "#8172B2"}
for p in [1, 2, 3, np.inf]:
    x, y = p_unit_ball_boundary(p)
    label = f"$p=\\infty$" if p == np.inf else f"$p={p}$"
    ax.plot(x, y, color=colors[p], lw=2.5, label=label)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_title("The $L^p$ unit ball's shape changes with $p$;\nonly $p=2$ is an actual (round) circle")
ax.legend()
plt.show()

print("Only p=2 satisfies the parallelogram law (the defining property of a norm that comes")
print("from an inner product) -- confirmed numerically in the Implementation section below.")
print("Visually, p=2 is the only shape here with no 'corners' and full rotational symmetry:")
print("every other L^p ball has flat edges (p=1, p=inf) or a non-circular rounded shape (p=3).")

## 🐍 Implementation from Scratch

We verify the triangle inequality holds for every $L^p$ norm but the parallelogram law holds
only for $p=2$ (confirming which norms are inner-product norms), build a genuine Cauchy sequence
whose limit escapes the rationals (the concrete meaning of "incomplete"), and confirm a matrix's
operator norm is the exact tight bound on how much it can stretch any vector.

In [ ]:
rng = np.random.default_rng(0)

def p_norm(v, p):
    if p == np.inf:
        return np.max(np.abs(v))
    return np.sum(np.abs(v) ** p) ** (1 / p)

u = rng.normal(size=5)
v = rng.normal(size=5)

print("Triangle inequality ||u+v|| <= ||u|| + ||v||, checked for several p (should ALWAYS hold):")
for p in [1, 2, 3, np.inf]:
    lhs, rhs = p_norm(u + v, p), p_norm(u, p) + p_norm(v, p)
    print(f"  p={p}: {lhs:.4f} <= {rhs:.4f}  (holds: {lhs <= rhs + 1e-10})")

print("\nParallelogram law ||u+v||^2 + ||u-v||^2 == 2||u||^2 + 2||v||^2 (should hold ONLY for p=2):")
for p in [1, 2, 3, np.inf]:
    lhs = p_norm(u + v, p) ** 2 + p_norm(u - v, p) ** 2
    rhs = 2 * p_norm(u, p) ** 2 + 2 * p_norm(v, p) ** 2
    print(f"  p={p}: LHS={lhs:.4f}, RHS={rhs:.4f}, match={np.isclose(lhs, rhs)}")
print("Only p=2 satisfies this -- exactly the norms that come from an inner product do.")

# --- Completeness: a Cauchy sequence of RATIONALS whose limit is IRRATIONAL ---
def rational_approx_sqrt2(n_digits):
    scale = 10 ** n_digits
    return np.floor(np.sqrt(2.0) * scale) / scale  # a genuine rational number (a terminating decimal)

seq = [rational_approx_sqrt2(n) for n in range(1, 10)]
diffs = [abs(seq[i+1] - seq[i]) for i in range(len(seq) - 1)]
print(f"\nCauchy sequence of RATIONAL approximations to sqrt(2):")
print(f"  successive differences (shrinking confirms it's Cauchy): {[f'{d:.1e}' for d in diffs]}")
print(f"  limit: {seq[-1]:.8f} vs true sqrt(2): {np.sqrt(2):.8f}")
print(f"  This Cauchy sequence of RATIONALS has an IRRATIONAL limit -- it escapes the space,")
print(f"  which is precisely what 'the rationals are not complete' means. L^p spaces are built")
print(f"  to be complete: every Cauchy sequence of functions in L^p has its limit IN L^p too.")

# --- Bounded linear operators: the operator norm as the tight amplification bound ---
A = rng.normal(size=(4, 3))
op_norm = np.linalg.svd(A, compute_uv=False)[0]  # largest singular value = operator norm

n_trials = 2000
ratios = []
for _ in range(n_trials):
    vv = rng.normal(size=3)
    ratios.append(np.linalg.norm(A @ vv) / np.linalg.norm(vv))
max_ratio_random = max(ratios)

U, S, Vt = np.linalg.svd(A)
v_star = Vt[0]  # the top right-singular vector -- should achieve the bound exactly
ratio_star = np.linalg.norm(A @ v_star) / np.linalg.norm(v_star)

print(f"\nOperator norm ||A|| (largest singular value): {op_norm:.4f}")
print(f"Max ||Av||/||v|| over {n_trials} random directions: {max_ratio_random:.4f}  (must stay <= ||A||: {max_ratio_random <= op_norm + 1e-9})")
print(f"Ratio at the theoretically optimal direction (top right-singular vector): {ratio_star:.6f}"
      f"  (equals ||A|| exactly: {np.isclose(ratio_star, op_norm)})")
print(f"This confirms ||A|| isn't just SOME valid bound -- it's the SMALLEST valid bound, achieved exactly.")

## 🤖 Why This Matters for AI

The **universal approximation theorem** is usually stated informally as "a wide-enough network
can approximate any function," but its precise, rigorous meaning is exactly this notebook's
language: for any target function $f^\star \in L^2$, there's a network $f_\theta$ making
$\|f_\theta - f^\star\|_2$ (an actual $L^2$-norm distance between two functions, not just a
training-set MSE) arbitrarily small. Below, we train networks of increasing width to fit a
target function using only 40 training points, then evaluate the true $L^2$ approximation error
against the *exact* target function on a fine, independent grid spanning the whole domain —
confirming the error shrinks with width in the sense this theorem actually promises, not merely
at the specific points used for training. This function-space view is also exactly what powers
**operator learning** (Fourier Neural Operators, DeepONet): rather than learning a
finite-dimensional function, these architectures learn a map between infinite-dimensional
function spaces directly — e.g. from a PDE's initial condition (a whole function) to its
solution at a later time (another whole function) — which only makes rigorous sense once you
have the $L^p$/Hilbert-space machinery this notebook built.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def target_fn(x):
    return np.sin(3 * x) + 0.3 * x**2

x_train = np.linspace(-2, 2, 40).reshape(-1, 1).astype(np.float32)
y_train = target_fn(x_train).astype(np.float32)

x_eval = np.linspace(-2, 2, 2000).reshape(-1, 1).astype(np.float32)  # fine, INDEPENDENT grid
y_eval_true = target_fn(x_eval).astype(np.float32)

class MLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, width), nn.Tanh(), nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 1))
    def forward(self, x):
        return self.net(x)

def l2_norm_continuous(f_vals, x_grid):
    # Approximates the CONTINUOUS L^2 norm ||f||_2 = sqrt(integral f(x)^2 dx) via the trapezoid rule
    return np.sqrt(np.trapezoid(f_vals.flatten() ** 2, x_grid.flatten()))

def train_and_measure(width, epochs=2000, lr=0.01):
    model = MLP(width)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x_t, y_t = torch.tensor(x_train), torch.tensor(y_train)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(x_t), y_t)
        loss.backward()
        opt.step()
    with torch.no_grad():
        y_pred_eval = model(torch.tensor(x_eval)).numpy()
    l2_error = l2_norm_continuous(y_pred_eval - y_eval_true, x_eval)
    return l2_error, model, y_pred_eval

widths = [4, 16, 64]
results = {}
for width in widths:
    l2_err, model, y_pred = train_and_measure(width)
    l2_target = l2_norm_continuous(y_eval_true, x_eval)
    results[width] = (l2_err, y_pred)
    print(f"width={width:3d}: L^2 approximation error ||f_theta - f*||_2 = {l2_err:.4f} "
          f"(relative to ||f*||_2={l2_target:.3f}: {l2_err/l2_target:.4f})")

print(f"\nError shrinks monotonically with width, measured in the TRUE L^2 norm over the whole")
print(f"domain (not just at the 40 training points) -- this is the universal approximation")
print(f"theorem's actual content, stated in this notebook's own language.")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(x_eval, y_eval_true, color='k', lw=2.5, label='target $f^*(x)$', zorder=5)
colors = {4: "#C44E52", 16: "#DD8452", 64: "#4C72B0"}
for width in widths:
    ax.plot(x_eval, results[width][1], color=colors[width], lw=1.5,
            label=f'width={width}, $\\|f_\\theta - f^*\\|_2$={results[width][0]:.3f}')
ax.scatter(x_train, y_train, color='k', s=15, alpha=0.4, zorder=4, label='training points')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title("Wider networks get closer to the target function in the\nactual $L^2$ norm, not just at the training points")
ax.legend(fontsize=9)
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $u=(1,1)$ and $v=(1,-1)$, compute $\|u\|_p$, $\|v\|_p$, $\|u+v\|_p$, and $\|u-v\|_p$ by
   hand for $p=1,2,\infty$, then check the parallelogram law
   $\|u+v\|_p^2+\|u-v\|_p^2=2\|u\|_p^2+2\|v\|_p^2$ at each $p$ as the dimensional/structural
   sanity check this construction calls for. At which $p$ does it hold?

<details>
<summary>💡 Solution</summary>

$u+v=(2,0)$ and $u-v=(0,2)$ for any $p$, so $\|u+v\|_p=\|u-v\|_p=2$ always, making the LHS
$2^2+2^2=8$ for every $p$. For the individual vectors, $\|u\|_p=\|v\|_p=(1^p+1^p)^{1/p}=2^{1/p}$,
so the RHS is $2\cdot2^{2/p}+2\cdot2^{2/p}=4\cdot2^{2/p}=2^{2+2/p}$.

| $p$ | LHS | RHS | Match? |
|---|---|---|---|
| $1$ | $8$ | $2^{4}=16$ | No |
| $2$ | $8$ | $2^{3}=8$ | **Yes** |
| $\infty$ | $8$ | $2^{2}=4$ | No |

Only $p=2$ satisfies the law ($2^{2+2/p}=8=2^3 \iff 2+2/p=3 \iff p=2$), matching the numerical
sweep in the Implementation section and confirming these two vectors already witness the general
fact.

</details>

### 💭 UNDERSTAND
2. Every linear operator on a *finite*-dimensional normed space is automatically bounded — the
   Implementation section's matrices can never illustrate an unbounded operator, no matter what
   numbers you put in them. Explain why finite dimensionality forces this, and name one place in
   the Theory section that already depends on it being false in general (i.e. on genuinely
   infinite-dimensional spaces like $L^2$ admitting *unbounded* linear operators).

<details>
<summary>💡 Solution</summary>

On a finite-dimensional space, the closed unit ball is compact (Heine–Borel), and a linear map
out of a finite-dimensional space is continuous — hence bounded — as a direct consequence: a
continuous image of a compact set is compact, which caps how much any single linear map can
stretch a unit vector. Riesz's theorem makes this a sharp dichotomy: the closed unit ball of a
normed space is compact *if and only if* the space is finite-dimensional, so once you pass to a
genuinely infinite-dimensional space like $L^2$, that compactness argument is no longer
available, and unbounded (discontinuous) linear operators provably exist. The Theory section's
own bounded-operator definition ("bounded if there's a constant $C$...") is only a substantive
restriction — something a linear operator on $L^2$ could fail to satisfy — because $L^2$ is
infinite-dimensional; on the finite matrices this notebook actually computes with, boundedness
was never in question, only the *value* of the (always-finite) operator norm was.

</details>

### ✏️ DERIVE
3. Prove that the parallelogram law holds **only** at $p=2$ among the $\ell^p$ norms on
   $\mathbb{R}^2$, using $u=(1,1)$, $v=(1,-1)$ as in COMPUTE, but this time symbolically for
   general $p\in[1,\infty)$ (do not just re-check the three numeric cases).

<details>
<summary>💡 Solution outline</summary>

As in COMPUTE, $u+v=(2,0)$, $u-v=(0,2)$, so for every finite $p\ge 1$,
$\|u+v\|_p=\|u-v\|_p=(2^p)^{1/p}=2$, independent of $p$ — the LHS of the parallelogram law is
identically $2^2+2^2=8$ for all $p$. Meanwhile $\|u\|_p=\|v\|_p=(1^p+1^p)^{1/p}=2^{1/p}$, so the
RHS is $2(2^{1/p})^2+2(2^{1/p})^2=4\cdot2^{2/p}=2^{2+2/p}$, a strictly *decreasing* function of
$p$ (the exponent $2+2/p$ strictly decreases as $p$ increases). Setting LHS $=$ RHS:
$$8=2^{2+2/p} \iff 3=2+2/p \iff p=2,$$
and because $2^{2+2/p}$ is strictly monotonic in $p$, this is the *unique* solution — for any
$p\ne 2$ the RHS is strictly on one side of $8$ (bigger for $p<2$, smaller for $p>2$), so the law
fails everywhere else, not just at the three sampled points. This is a special case of the
Jordan–von Neumann theorem: a norm is induced by an inner product if and only if it satisfies
the parallelogram law for *all* vectors, and this computation exhibits one concrete pair $(u,v)$
that already rules out every $p\ne 2$.

</details>

### 🐍 IMPLEMENT
4. Implement power iteration to estimate a matrix's operator norm — repeatedly apply $A^\top A$
   to a vector, renormalize, and take the square root of the Rayleigh quotient
   $x^\top(A^\top A)x$ — and track the estimate's error against `np.linalg.svd`'s top singular
   value after each iteration. Run it on both the notebook's original random $4\times3$ `A` and
   a constructed large-gap matrix (singular values $[10, 0.1, 0.05]$, via
   `U @ np.diag([10, 0.1, 0.05]) @ Vt[:3]` with random orthogonal `U`, `Vt` from `np.linalg.qr`).

<details>
<summary>✅ How to know you're right</summary>

Both runs should converge to `np.linalg.svd(A, compute_uv=False)[0]` to within `1e-6` or better
given enough iterations — that agreement between two independent numerical routes (power
iteration vs. full SVD) is the actual check, not a printed number. The *rate* should differ
sharply between the two matrices: power iteration's convergence is governed by
$(\sigma_2/\sigma_1)^k$, so the large-gap matrix ($\sigma_2/\sigma_1=0.01$) should already be
accurate to below $10^{-4}$ after a single iteration, while the original matrix
($\sigma_2/\sigma_1\approx0.41$) needs several iterations (around 3) to reach the same tolerance
— a big spectral gap makes power iteration converge fast, precisely because the "runner-up"
direction's contribution is suppressed geometrically every iteration.

</details>

### 🤖 APPLY
5. Take one of the trained `MLP` models from the "Why This Matters for AI" section (e.g.
   `width=16`). Compute a Lipschitz bound on the whole network by multiplying together the
   operator norms (largest singular values) of its three `nn.Linear` weight matrices, then
   estimate the network's actual (empirical) Lipschitz constant by sampling many random pairs
   $x_1,x_2\in[-2,2]$ and computing $\max |f_\theta(x_1)-f_\theta(x_2)|/|x_1-x_2|$.

<details>
<summary>✅ What you should observe</summary>

The empirical Lipschitz constant should always stay *below* the product-of-operator-norms
bound — that inequality is guaranteed by submultiplicativity of the operator norm through
composition, and `Tanh`'s own Lipschitz constant of 1 not increasing anything further. In
practice the bound is quite loose: for a trained `width=16` or `width=64` model here, the
empirical constant comes out around 3.7-3.8, while the naive product-of-operator-norms bound is
roughly 10x larger (around 36-40) — the gap exists because the bound assumes every layer's
worst-case stretch direction lines up with the next layer's, which essentially never happens
for an actually-trained network. This is exactly the phenomenon behind why naive Lipschitz
bounds on real trained networks (used e.g. in certified-robustness or generalization arguments)
are typically very pessimistic compared to a network's true behavior.

</details>

### 🚀 CHALLENGE
6. Change `target_fn` in the AI section to a discontinuous function, e.g. `np.sign(x)`, retrain
   at the same widths (4, 16, 64), and compare against the smooth-target results. First settle
   the basic question: is `np.sign(x)` restricted to $[-2,2]$ actually *in* $L^2([-2,2])$ at
   all? Then investigate whether the $L^2$ approximation error still shrinks toward 0 as width
   grows the way it did for the smooth target, and connect what you find to `MLP`'s use of
   smooth `Tanh` activations.

<details>
<summary>🔍 What a strong answer covers</summary>

Yes — $\text{sign}(x)$ is trivially in $L^2([-2,2])$: it's a bounded function ($|\text{sign}(x)|
\le 1$) on a bounded domain, so $\int_{-2}^{2}|\text{sign}(x)|^2\,dx=4<\infty$ regardless of the
jump at $x=0$; $L^2$ membership never required continuity. The more interesting finding is
empirical: under this notebook's exact training setup (40 fixed training points, `Tanh` MLPs,
Adam, a few thousand epochs), the $L^2$ error for the smooth target drops sharply and
monotonically with width (roughly $0.048\to0.009\to0.004$ at widths $4,16,64$), but for
`np.sign(x)` the error does *not* shrink the same way — it plateaus in a narrow band
(empirically around $0.14$-$0.19$) across widths 4 through 256 and across random seeds, and the
worst-case pointwise error near the jump can even *exceed* 1 (overshoot), reminiscent of the
Gibbs phenomenon in Fourier approximation of discontinuities. A strong answer notices this is not
a contradiction of universal approximation (which still guarantees arbitrarily small $L^2$ error
*given enough width, data, and training*) but a demonstration that a fixed, small training
budget interacts badly with a jump discontinuity specifically because every `Tanh` unit is
smooth: approximating a vertical jump forces increasingly steep (large-weight) sigmoids clustered
right at $x=0$, which is a much harder optimization landscape than fitting a smooth target, so
the *practical* convergence-with-width story can look qualitatively different even when the
*theoretical* approximation-theory story (sign(x) is approximable in $L^2$) does not change.

</details>

---

## 📚 Further Reading
- Kreyszig, *Introductory Functional Analysis with Applications*, Ch. 1-2
- Hornik, Stinchcombe & White, [\"Multilayer feedforward networks are universal approximators\"](https://www.sciencedirect.com/science/article/abs/pii/0893608089900208)
- Lu et al., [\"Learning nonlinear operators via DeepONet\"](https://arxiv.org/abs/1910.03193)

---

*Next notebook: [Kernel Methods and RKHS](02_kernel_methods_rkhs.ipynb)*